In [2]:
import requests
import pandas as pd
from tqdm import tqdm
import sqlite3
import random
import re
import hashlib

In [4]:
url = "https://api.tcgdex.net/v2/en/cards"

response_all_cards = requests.get(url, timeout=15)
response_all_cards.raise_for_status()

response_all_cards_json = response_all_cards.json()

In [5]:
print("Total number of cards:", len(response_all_cards_json))
print("Sample card data:", response_all_cards_json[3])

Total number of cards: 23160
Sample card data: {'id': 'swsh9-001', 'localId': '001', 'name': 'Exeggcute', 'image': 'https://assets.tcgdex.net/en/swsh/swsh9/001'}


In [5]:
def get_card_details(card_id):
    url = f"https://api.tcgdex.net/v2/en/cards/{card_id}"
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    return response.json()


card_details = get_card_details(response_all_cards_json[3]["id"])
print("Card details:", card_details)

Card details: {'category': 'Pokemon', 'id': 'swsh9-001', 'image': 'https://assets.tcgdex.net/en/swsh/swsh9/001', 'localId': '001', 'name': 'Exeggcute', 'rarity': 'Common', 'set': {'cardCount': {'official': 172, 'total': 216}, 'id': 'swsh9', 'logo': 'https://assets.tcgdex.net/en/swsh/swsh9/logo', 'name': 'Brilliant Stars', 'symbol': 'https://assets.tcgdex.net/univ/swsh/swsh9/symbol'}, 'variants': {'firstEdition': False, 'holo': False, 'normal': True, 'reverse': True, 'wPromo': False}, 'variants_detailed': [{'type': 'normal', 'size': 'standard', 'variantId': 'generated'}, {'type': 'reverse', 'size': 'standard', 'variantId': 'generated'}], 'dexId': [102], 'hp': 50, 'types': ['Grass'], 'stage': 'Basic', 'attacks': [{'cost': ['Colorless'], 'name': 'Ram', 'damage': 10}, {'cost': ['Grass', 'Colorless'], 'name': 'Seed Bomb', 'damage': 20}], 'retreat': 1, 'regulationMark': 'F', 'legal': {'standard': False, 'expanded': True}, 'updated': '2025-08-16T20:39:55Z', 'pricing': {'cardmarket': {'updated

In [8]:
# JSON (dict) -> DataFrame "piatto" (1 riga)
df_card = pd.json_normalize(card_details, sep="_")
display(df_card)

,category,id,image,localId,name,rarity,variants_detailed,dexId,hp,types,...,pricing_cardmarket_avg1,pricing_cardmarket_avg7,pricing_cardmarket_avg30,pricing_cardmarket_avg-holo,pricing_cardmarket_low-holo,pricing_cardmarket_trend-holo,pricing_cardmarket_avg1-holo,pricing_cardmarket_avg7-holo,pricing_cardmarket_avg30-holo,pricing_tcgplayer
0,Pokemon,swsh9-001,https://assets.tcgdex.net/en/swsh/swsh9/001,001,Exeggcute,Common,"[{'type': 'normal', 'size': 'standard', 'varia...",[102],50,[Grass],...,0.02,0.03,0.03,0.16,0.02,0.09,0.05,0.19,0.16,None


In [6]:
all_df_cards = []
for card in tqdm(response_all_cards_json[:300]):
    try:
        details = get_card_details(card["id"])
        df_card = pd.json_normalize(details, sep="_")
        df_card["espansione_id"] = details["set"]["id"]
        df_card["espansione_nome"] = details["set"]["name"]
        if "image" not in details:
            df_card["image"] = [None]
        if not details["pricing"]["cardmarket"]:
            df_card["pricing_cardmarket_low"] = [0]
        df_card = df_card[
            [
                "id",
                "name",
                "espansione_id",
                "espansione_nome",
                "pricing_cardmarket_low",
                "image",
            ]
        ]
        all_df_cards.append(df_card)
    except Exception as e:
        print(f"Error fetching details for card {card['id']}: {e}")
        # break

  1%|          | 2/300 [00:00<02:01,  2.46it/s]

Error fetching details for card exu-%3F: 404 Client Error: Not Found for url: https://api.tcgdex.net/v2/en/cards/exu-%3F


100%|██████████| 300/300 [01:23<00:00,  3.60it/s]


In [7]:
conn = sqlite3.connect("../card_database.db")
cursor = conn.cursor()

df_all = pd.concat(all_df_cards, ignore_index=True)
df_all.to_sql(
    "DatabaseCards",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)

299

### Generate stock example

In [8]:
def generate_barcode(nome, espansione, condizione):
    def clean(s):
        return re.sub(r"[^A-Z0-9]", "", s.upper())

    # parte leggibile
    base = f"{clean(nome)[:4]}-{clean(espansione)[:3]}-{clean(condizione)[:2]}"

    # hash deterministico
    raw = f"{nome}|{espansione}|{condizione}".upper()
    short_hash = hashlib.md5(raw.encode()).hexdigest()[:6].upper()

    return f"{base}-{short_hash}"


generate_barcode("Pikachu V Full Art", "SWSH039", "Excellent")

'PIKA-SWS-EX-4BF6D1'

In [15]:
df_all = pd.concat(all_df_cards, ignore_index=True)
df_test = df_all.sample(15)[["id", "name", "espansione_id", "espansione_nome"]].copy()
cards_condizioni = [
    "Mint",
    "Near Mint",
    "Excellent",
    "Good",
    "Light Played",
    "Played",
    "Poor",
]
df_test["condizione"] = [random.choice(cards_condizioni) for _ in range(len(df_test))]
df_test["barcode"] = df_test.apply(
    lambda row: generate_barcode(
        row["name"], row["espansione_id"], row["condizione"]
    ),
    axis=1,
)
df_test["prezzo"] = [round(random.uniform(1, 100), 2) for _ in range(len(df_test))]
df_test["quantita_stock"] = [random.randint(1, 5) for _ in range(len(df_test))]
df_test["prezzo_acquisto"] = [
    round(random.uniform(0, 1) * prezzo, 2) for prezzo in df_test["prezzo"]
]
df_test

,id,name,espansione_id,espansione_nome,condizione,barcode,prezzo,quantita_stock,prezzo_acquisto
142,sv10.5b-001,Snivy,sv10.5b,Black Bolt,Light Played,SNIV-SV1-LI-207C7C,80.92,2,56.78
278,hgss1-2,Azumarill,hgss1,HeartGold SoulSilver,Played,AZUM-HGS-PL-A1FC3A,31.37,1,11.92
186,P-A-001,Potion,P-A,Promos-A,Played,POTI-PA-PL-431D84,84.65,5,31.91
270,sm12-2,Oddish,sm12,Cosmic Eclipse,Near Mint,ODDI-SM1-NE-4FD985,68.12,5,14.52
239,tk-xy-su-2,Water Energy,tk-xy-su,XY trainer Kit (Suicune),Light Played,WATE-TKX-LI-8F8AFC,23.28,3,0.85
108,neo3-1,Ampharos,neo3,Neo Revelation,Poor,AMPH-NEO-PO-0EB041,57.16,1,36.62
166,hgss4-1,Aggron,hgss4,Triumphant,Good,AGGR-HGS-GO-92F7EE,44.01,1,19.45
60,bw4-1,Pinsir,bw4,Next Destinies,Mint,PINS-BW4-MI-C5C1FF,60.55,1,5.06
144,ex1-1,Aggron,ex1,Ruby & Sapphire,Good,AGGR-EX1-GO-80FD8A,43.89,1,6.70
129,ex5-1,Banette,ex5,Hidden Legends,Near Mint,BANE-EX5-NE-2AE474,35.17,5,12.37


In [16]:
conn = sqlite3.connect("../pokemon.db")
cursor = conn.cursor()
df_test.to_sql(
    "stock",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()